# Agentic SOC Gemini Enterprise -- Deployment Notebook

This notebook deploys the SOC Manager (orchestrator) agent engine and links it to a Gemini Enterprise (AgentSpace) app.

The repository contains a five-agent fleet: `agent_soc_manager` (orchestrator) plus four A2A specialists (`agent_a2a_tier2`, `agent_a2a_threat_hunter`, `agent_a2a_cti_researcher`, `agent_a2a_detection_engineer`). This notebook deploys the orchestrator; each specialist deploys the same way by passing `--agent-module <name>` in step 4.

## Prerequisites
- A Google Cloud Project with billing enabled.
- Necessary APIs enabled (Vertex AI, Discovery Engine, etc.).
- Proper IAM permissions.


## 1. Setup Environment
Clone the repository and install dependencies.

In [ ]:
!git clone --recurse-submodules https://github.com/dandye/agentic_soc_gemini_enterprise.git
%cd agentic_soc_gemini_enterprise
!pip install -r requirements.txt
!pip install google-cloud-aiplatform google-cloud-discoveryengine google-cloud-storage google-cloud-resource-manager


## 2. Authenticate with Google Cloud
Authenticate your Google Cloud account to access resources.

In [ ]:
from google.colab import auth


auth.authenticate_user()
print('Authenticated')

## 3. Configuration
Enter your project details and configuration parameters.
These will be saved to a `.env` file for the deployment scripts to use.

In [ ]:
import os


# Prompt for user input
project_id = input("Enter your Google Cloud Project ID: ")
project_number = input("Enter your Google Cloud Project Number: ")
location = input("Enter your Google Cloud Location (e.g., us-central1): ")
staging_bucket = input("Enter your GCS Staging Bucket (gs://...): ")

# Optional inputs for specific integrations
chronicle_project_id = input("Enter Chronicle Project ID (optional): ")
chronicle_customer_id = input("Enter Chronicle Customer ID (optional): ")
chronicle_sa_path = input("Enter Path to Chronicle Service Account JSON (optional): ")
soar_url = input("Enter SOAR URL (optional): ")
soar_api_key = input("Enter SOAR API Key (optional): ")
gti_api_key = input("Enter Google Threat Intelligence API Key (optional): ")
rag_corpus_id = input("Enter RAG Corpus ID (optional, e.g., projects/.../ragCorpora/...): ")

# Create .env file content
env_content = f"""
GCP_PROJECT_ID={project_id}
GCP_PROJECT_NUMBER={project_number}
GCP_LOCATION={location}
GCP_STAGING_BUCKET={staging_bucket}
"""

if chronicle_project_id:
    env_content += f"CHRONICLE_PROJECT_ID={chronicle_project_id}\n"
if chronicle_customer_id:
    env_content += f"CHRONICLE_CUSTOMER_ID={chronicle_customer_id}\n"
if chronicle_sa_path:
    env_content += f"CHRONICLE_SERVICE_ACCOUNT_PATH={chronicle_sa_path}\n"
if soar_url:
    env_content += f"SOAR_URL={soar_url}\n"
if soar_api_key:
    env_content += f"SOAR_API_KEY={soar_api_key}\n"
if gti_api_key:
    env_content += f"GTI_API_KEY={gti_api_key}\n"
if rag_corpus_id:
    env_content += f"RAG_CORPUS_ID={rag_corpus_id}\n"

# Write to .env file
with open(".env", "w") as f:
    f.write(env_content)

print(".env file created successfully.")
# Set environment variables in the current session as well
os.environ.update({
    "GCP_PROJECT_ID": project_id,
    "GCP_LOCATION": location,
    "GCP_STAGING_BUCKET": staging_bucket,
})


## 4. Deploy Agent Engine
Deploy the agent to Vertex AI Agent Engine. This step may take a few minutes.

In [ ]:
# Deploy the orchestrator agent engine (default --agent-module is agent_soc_manager)
!python manage.py agent-engine create --agent-module agent_soc_manager

# To deploy the A2A specialists as their own engines, repeat with e.g.:
#   !python manage.py agent-engine create --agent-module agent_a2a_threat_hunter

# The resource name is printed in the deploy output; paste it below so the
# .env is updated for the register/verify steps.
agent_engine_resource_name = input("Please copy the AGENT_ENGINE_RESOURCE_NAME from the output above and paste it here: ")

with open(".env", "a") as f:
    f.write(f"\nAGENT_ENGINE_RESOURCE_NAME={agent_engine_resource_name}\n")

os.environ["AGENT_ENGINE_RESOURCE_NAME"] = agent_engine_resource_name
print(f"Updated .env with AGENT_ENGINE_RESOURCE_NAME={agent_engine_resource_name}")


## 5. Create AgentSpace App (Optional)
If you don't have an AgentSpace app yet, create one now.
**Note:** To ensure the app appears in the Gemini Enterprise web UI, we use specific flags (`--app-type APP_TYPE_INTRANET`, `--industry-vertical GENERIC`).

In [ ]:
create_new_app = input("Do you want to create a new AgentSpace app? (y/n): ").lower() == 'y'

if create_new_app:
    app_name = input("Enter a name for the app (default: SOC Agent App): ") or "SOC Agent App"
    # Create the app
    !python manage.py agentspace create-app --name "{app_name}" --type SOLUTION_TYPE_CHAT --no-datastore --app-type APP_TYPE_INTRANET --industry-vertical GENERIC
else:
    app_id = input("Enter your existing AgentSpace App ID: ")
    if app_id:
        with open(".env", "a") as f:
            f.write(f"\nAGENTSPACE_APP_ID={app_id}\n")
        os.environ["AGENTSPACE_APP_ID"] = app_id
        print(f"Updated .env with AGENTSPACE_APP_ID={app_id}")

## 6. Link Agent to AgentSpace
Register the deployed agent engine with the AgentSpace app.

In [ ]:
!python manage.py agentspace register

## 7. Verify Deployment
Verify that the agent is correctly registered and get the UI URL.

In [ ]:
!python manage.py agentspace verify
!python manage.py agentspace url